# Causal Machine Learning for Policy Evaluation
## A hands-on Colab walk-through

This notebook is a runnable companion to the blog post [*Causal Machine Learning for Policy Evaluation: From ATE to IATE to a Better Assignment Rule*](https://carlos-mendez.org/post/python_cml/).

You will work through the full CML roadmap on a **synthetic Flemish-ALMP-style cohort of 5,000 jobseekers**:

1. **ATE** -- the population-average effect of a job-training programme, via DoubleML
2. **GATE** -- the effect by Dutch language proficiency, via doubly-robust pseudo-outcomes
3. **IATE** -- per-individual effects, via EconML's CausalForestDML
4. **Policy** -- a welfare-maximising training-assignment rule

Because the data are synthetic, the *true* individual treatment effect is known for every row, so every estimator can be benchmarked against the truth.

> **How to use this notebook:** click `Runtime → Run all` (or press `Ctrl+F9`) to execute every cell from top to bottom. The full pipeline takes about 90 seconds on a free Colab CPU runtime. No GPU needed.


In [ ]:
!pip install doubleml econml --quiet

## Setup

Import the stack and fix the random seed. Setting `np.random.seed(RANDOM_SEED)` is *not* dead code: DoubleML's internal cross-fit splitter uses the legacy global numpy RNG, so removing this line makes the ATE estimate drift across runs.


In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import LogisticRegression

from doubleml import DoubleMLData, DoubleMLIRM
from econml.dml import CausalForestDML

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Site palette (light theme)
COLOR_BLUE   = "#6a9bcc"
COLOR_ORANGE = "#d97757"
COLOR_BLACK  = "#141413"
COLOR_TEAL   = "#00d4c8"
COLOR_GRAY   = "#999999"

X_COLS = ["age", "edu_years", "prior_emp_months", "dutch_prof", "female", "migrant"]


## Data: a synthetic ALMP cohort

We simulate a Flemish-ALMP-style cohort directly inside the notebook so it runs on a fresh Colab session with no external dependencies. The data-generating process (DGP) records six pre-treatment covariates ($X$) for each jobseeker -- age, education, prior employment, Dutch proficiency on a 0-3 scale, sex, and migrant status -- and uses them to drive both:

- **the propensity to receive training** (selection on observables -- low-Dutch-proficiency jobseekers are steered into training by their caseworkers), and
- **the outcome** (months of employment in a 30-month window) plus the **individual treatment effect** $\tau_i$ (which declines with Dutch proficiency).

Because we simulate the data ourselves, we know the true $\tau_i$ for every row and can benchmark every estimator against the truth.


In [ ]:
def simulate_almp(n=5000, seed=RANDOM_SEED):
    # Generate a synthetic Flanders-ALMP-style cohort with known truth.
    rng = np.random.default_rng(seed)

    # Pre-treatment covariates X
    age              = rng.uniform(20, 60, size=n)
    edu_years        = np.clip(rng.normal(12, 3, size=n), 6, 20)
    prior_emp_months = rng.beta(2, 5, size=n) * 60.0
    dutch_prof       = rng.choice([0, 1, 2, 3], size=n, p=[0.25, 0.30, 0.30, 0.15])
    female           = rng.binomial(1, 0.48, size=n)
    migrant          = rng.binomial(1, 0.30, size=n)

    # Propensity score: lower Dutch proficiency -> higher probability of training
    logit_pi = (-0.6
                + 0.020 * (40 - age)
                + 0.05  * (12 - edu_years)
                + 0.015 * (30 - prior_emp_months)
                + 0.30  * (3 - dutch_prof)
                + 0.20  * migrant
                - 0.10  * female)
    pi_true = np.clip(1.0 / (1.0 + np.exp(-logit_pi)), 0.05, 0.95)
    D = rng.binomial(1, pi_true)

    # Individual treatment effect: declines with Dutch proficiency
    tau = (3.0
           + 1.5 * (3 - dutch_prof)
           + 0.4 * migrant
           - 0.02 * (age - 40)
           + rng.normal(0, 0.3, size=n))

    # Baseline outcome Y(0): months employed
    Y0 = (12.0
          + 0.20 * prior_emp_months
          + 0.30 * edu_years
          + 1.0  * dutch_prof
          - 0.05 * (age - 40) ** 2 / 10
          + rng.normal(0, 2.5, size=n))
    Y0 = np.clip(Y0, 0, 30)
    Y1 = np.clip(Y0 + tau, 0, 30)
    Y_obs = D * Y1 + (1 - D) * Y0

    obs = pd.DataFrame({
        "age": age,
        "edu_years": edu_years,
        "prior_emp_months": prior_emp_months,
        "dutch_prof": dutch_prof.astype(int),
        "female": female.astype(int),
        "migrant": migrant.astype(int),
        "D": D.astype(int),
        "Y": Y_obs,
    })
    truth = pd.DataFrame({
        "Y0": Y0, "Y1": Y1, "tau": tau, "pi_true": pi_true,
        "dutch_prof": dutch_prof.astype(int),
    })
    return obs, truth


df, truth = simulate_almp(n=5000, seed=RANDOM_SEED)

print(f"Sample size            : {len(df):,}")
print(f"Treatment share P(D=1) : {df['D'].mean():.3f}")
print(f"Mean outcome E[Y]      : {df['Y'].mean():.2f} months employed (out of 30)")
print()
print("First 5 rows of observed data:")
df.head()


Compute the **true ATE** and **true GATE by Dutch proficiency** analytically. The true GATE declines monotonically with Dutch proficiency: the lowest-proficiency stratum benefits roughly 2.4× more from training than the highest. This is the kind of heterogeneity CML methods are designed to recover.


In [ ]:
true_ate  = float(truth["tau"].mean())
true_gate = truth.groupby("dutch_prof")["tau"].mean()

print(f"True ATE                  : {true_ate:.3f}")
for z in [0, 1, 2, 3]:
    print(f"True GATE(dutch_prof={z})  : {true_gate.loc[z]:.3f}")


## Estimands

We target three estimands of increasing granularity:

- **ATE** $= E[Y(1) - Y(0)]$ -- population-average effect
- **GATE**$(z) = E[Y(1) - Y(0) \mid Z = z]$ -- average effect within a subgroup
- **IATE**$(x) = E[Y(1) - Y(0) \mid X = x]$ -- individual effect at every covariate profile

The framing is **observational** -- we assume *unconfoundedness* (selection on observables). The naive difference-in-means is therefore biased; CML methods earn their keep by addressing that confounding through flexible nuisance estimators and orthogonal scores.

## Step 1 -- Overlap diagnostic

Causal estimation requires every covariate profile to have a non-trivial chance of being treated *and* untreated; otherwise the model has to extrapolate. We fit a logistic propensity score and check that the treated/untreated histograms overlap. The logistic regression here is purely for visualisation -- DoubleML and CausalForestDML will fit their own nuisance models later.


In [ ]:
ps_lr  = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED).fit(df[X_COLS], df["D"])
ps_hat = ps_lr.predict_proba(df[X_COLS])[:, 1]

print(f"Propensity range          : [{ps_hat.min():.3f}, {ps_hat.max():.3f}]")
print(f"P(D=1 | X) mean (treated) : {ps_hat[df['D']==1].mean():.3f}")
print(f"P(D=1 | X) mean (untreat.): {ps_hat[df['D']==0].mean():.3f}")

fig, ax = plt.subplots(figsize=(8.5, 4.5))
bins = np.linspace(0, 1, 31)
ax.hist(ps_hat[df["D"] == 0], bins=bins, alpha=0.65, color=COLOR_BLUE,
        label="Untreated (D=0)", edgecolor="white")
ax.hist(ps_hat[df["D"] == 1], bins=bins, alpha=0.65, color=COLOR_ORANGE,
        label="Treated (D=1)",  edgecolor="white")
ax.set_xlabel(r"Estimated propensity score $\hat{\pi}(X)$")
ax.set_ylabel("Number of individuals")
ax.set_title("Covariate overlap: propensity-score distribution by treatment status")
ax.legend()
plt.tight_layout()
plt.show()


Propensities fall safely inside [0.21, 0.81]. Both groups overlap heavily, so positivity holds. Now we can move on to the simplest possible estimator and watch it fail.

## Step 2 -- Naive baseline (biased!)

The simplest estimator: average $Y$ for the treated, average $Y$ for the untreated, subtract. Under randomized assignment this would be unbiased. Under observational data with selection on observables, it generally is not.


In [ ]:
y_treated   = df.loc[df["D"] == 1, "Y"].mean()
y_untreated = df.loc[df["D"] == 0, "Y"].mean()
naive_ate   = y_treated - y_untreated

n1, n0 = int((df["D"] == 1).sum()), int((df["D"] == 0).sum())
s1, s0 = df.loc[df["D"] == 1, "Y"].var(ddof=1), df.loc[df["D"] == 0, "Y"].var(ddof=1)
naive_se = float(np.sqrt(s1 / n1 + s0 / n0))

print(f"True ATE       : {true_ate:.3f}")
print(f"Naive estimate : {naive_ate:.3f} "
      f"[95% CI {naive_ate - 1.96 * naive_se:.3f}, {naive_ate + 1.96 * naive_se:.3f}]")
print(f"Bias           : {naive_ate - true_ate:+.3f} months "
      f"({100*(naive_ate - true_ate)/true_ate:+.1f}% of truth)")


The naive estimate (~5.11) is biased downward by ~0.52 months and its 95% CI does **not** cover the truth (5.628). Why? Caseworkers steer the people with the largest effects into training, and the naive estimator can't disentangle that selection.

## Step 3 -- ATE via Double Machine Learning

`DoubleMLIRM` implements the **Interactive Regression Model**: a cross-fitted, doubly-robust estimator of the ATE under unconfoundedness. The doubly-robust score for observation $i$ uses two nuisance functions -- an outcome regression $g(d, X) = E[Y \mid D = d, X]$ and a propensity score $m(X) = P(D = 1 \mid X)$:

$$\psi_i = g_1(X_i) - g_0(X_i) + \frac{D_i \, (Y_i - g_1(X_i))}{m(X_i)} - \frac{(1 - D_i) \, (Y_i - g_0(X_i))}{1 - m(X_i)}$$

It's "doubly robust" because $E[\psi_i] = $ ATE as long as **either** $g$ **or** $m$ is correctly specified. We fit it with random-forest nuisances and 5-fold cross-fitting.


In [ ]:
dml_data = DoubleMLData(df, y_col="Y", d_cols="D", x_cols=X_COLS)

ml_g = RandomForestRegressor(n_estimators=200, max_features="sqrt",
                             min_samples_leaf=5, random_state=RANDOM_SEED, n_jobs=-1)
ml_m = RandomForestClassifier(n_estimators=200, max_features="sqrt",
                              min_samples_leaf=5, random_state=RANDOM_SEED, n_jobs=-1)

dml_irm = DoubleMLIRM(
    dml_data, ml_g=ml_g, ml_m=ml_m,
    n_folds=5, score="ATE", trimming_threshold=0.01,
)
dml_irm.fit(store_predictions=True)

ate_dml = float(dml_irm.coef[0])
ci      = dml_irm.confint(level=0.95).iloc[0]
ci_low, ci_high = float(ci.iloc[0]), float(ci.iloc[1])

print(f"True ATE            : {true_ate:.3f}")
print(f"DoubleML ATE        : {ate_dml:.3f} [95% CI {ci_low:.3f}, {ci_high:.3f}]")
print(f"95% CI covers truth : {bool(ci_low <= true_ate <= ci_high)}")
print(f"Bias closed         : "
      f"{100*(1 - abs(ate_dml-true_ate)/abs(naive_ate-true_ate)):.0f}% of naive bias")


DoubleML closes about 79% of the naive bias and produces the only 95% CI that covers the truth.

## Step 4 -- GATE by Dutch proficiency

A policymaker wants the next layer down: does the effect depend on who the jobseeker is? We extract the cross-fitted nuisance predictions DoubleML already stored, compute the doubly-robust pseudo-outcome $\psi_i$ for every individual, and average it within each Dutch-proficiency stratum. Because $E[\psi_i \mid Z_i = z] = \text{GATE}(z)$, this is an unbiased estimator of each stratum's effect.


In [ ]:
preds = dml_irm.predictions
g0 = np.asarray(preds["ml_g0"]).squeeze()
g1 = np.asarray(preds["ml_g1"]).squeeze()
m  = np.asarray(preds["ml_m"]).squeeze()

y_arr, d_arr = df["Y"].values, df["D"].values
psi = (g1 - g0
       + d_arr * (y_arr - g1) / m
       - (1 - d_arr) * (y_arr - g0) / (1 - m))

rows = []
for z in [0, 1, 2, 3]:
    mask  = (df["dutch_prof"] == z).values
    psi_z = psi[mask]
    est   = float(psi_z.mean())
    se    = float(psi_z.std(ddof=1) / np.sqrt(mask.sum()))
    rows.append({"dutch_prof": z, "n": int(mask.sum()),
                 "gate_est": est, "se": se,
                 "ci_low": est - 1.96 * se, "ci_high": est + 1.96 * se,
                 "truth": float(true_gate.loc[z])})
gate_df = pd.DataFrame(rows)
print(gate_df.to_string(index=False, float_format=lambda v: f"{v:7.3f}"))

# Bar chart
fig, ax = plt.subplots(figsize=(9, 5))
x_pos = np.arange(4)
width = 0.36
ax.bar(x_pos - width/2, gate_df["gate_est"], width,
       yerr=1.96 * gate_df["se"], capsize=5,
       color=COLOR_BLUE, edgecolor=COLOR_BLACK, label="DoubleML GATE (95% CI)")
ax.bar(x_pos + width/2, gate_df["truth"], width,
       color=COLOR_ORANGE, edgecolor=COLOR_BLACK, alpha=0.85, label="True GATE")
ax.set_xticks(x_pos)
ax.set_xticklabels(["0 (no)", "1 (low)", "2 (intermediate)", "3 (native)"])
ax.set_xlabel("Dutch language proficiency")
ax.set_ylabel("GATE — extra months of employment")
ax.set_title("Group Average Treatment Effect by Dutch proficiency")
ax.legend()
plt.tight_layout()
plt.show()


The estimated bars track the true bars almost exactly, with every stratum estimate within 0.22 months of its true target. The lowest-proficiency stratum benefits roughly 2.6× more from training than the highest.

## Step 5 -- IATE via Causal Forest DML

The GATE collapses every jobseeker in a stratum into a single number. Two people with the same `dutch_prof` value can still differ in age, education, prior employment, and migrant status, and the programme might help them very differently. The **Individual Average Treatment Effect** $\tau(x) = E[Y(1) - Y(0) \mid X = x]$ asks for a separate prediction at every covariate profile.

`CausalForestDML` is a Python implementation of the [generalized random forest](https://doi.org/10.1214/18-AOS1709) framework of Athey, Tibshirani & Wager (2019). The trees split on heterogeneity in the **treatment effect** rather than on heterogeneity in the outcome -- every leaf becomes a small neighbourhood within which the IATE is locally constant.


In [ ]:
cf = CausalForestDML(
    model_y=RandomForestRegressor(n_estimators=200, min_samples_leaf=5,
                                  random_state=RANDOM_SEED, n_jobs=-1),
    model_t=RandomForestClassifier(n_estimators=200, min_samples_leaf=5,
                                   random_state=RANDOM_SEED, n_jobs=-1),
    discrete_treatment=True,
    n_estimators=400, min_samples_leaf=15, max_samples=0.5,
    random_state=RANDOM_SEED, n_jobs=-1,
)
X_arr = df[X_COLS].values
cf.fit(df["Y"].values, df["D"].values, X=X_arr)

iate_hat = np.asarray(cf.effect(X_arr)).ravel()
mae  = float(np.abs(iate_hat - truth["tau"].values).mean())
corr = float(np.corrcoef(iate_hat, truth["tau"].values)[0, 1])

print(f"True ATE                : {true_ate:.3f}")
print(f"Mean of estimated IATEs : {iate_hat.mean():.3f}")
print(f"MAE(IATE, truth)        : {mae:.3f}")
print(f"Corr(IATE, truth)       : {corr:.3f}")

# Scatter of IATE vs truth
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(truth["tau"].values, iate_hat, s=6, alpha=0.35,
           color=COLOR_BLUE, edgecolor="none", label="Individuals")
lim_low, lim_high = -1, 11
ax.plot([lim_low, lim_high], [lim_low, lim_high], color=COLOR_ORANGE,
        linewidth=1.8, label="45° (perfect estimation)")
ax.set_xlabel(r"True individual effect $\tau_i$")
ax.set_ylabel(r"Causal Forest estimate $\hat{\tau}_i$")
ax.set_title(f"Estimated vs true IATE — corr = {corr:.3f}, MAE = {mae:.2f}")
ax.legend()
ax.set_aspect("equal", adjustable="box")
plt.tight_layout()
plt.show()


Correlation **0.956** with the truth and MAE **0.40 months** -- small relative to the 4.5-month spread of true effects. The forest produces usable rankings AND well-calibrated levels.

## Step 6 -- Method comparison

We line up all three ATE estimators against the truth in a single forest plot.


In [ ]:
cf_mean = float(iate_hat.mean())
cf_se   = float(iate_hat.std(ddof=1) / np.sqrt(len(iate_hat)))

comp = pd.DataFrame({
    "method":   ["Naive (DiM)", "DoubleML (IRM)",
                 "CausalForestDML (mean of IATEs)", "Truth"],
    "estimate": [naive_ate, ate_dml, cf_mean, true_ate],
    "ci_low":   [naive_ate - 1.96*naive_se, ci_low,
                 cf_mean - 1.96*cf_se, true_ate],
    "ci_high":  [naive_ate + 1.96*naive_se, ci_high,
                 cf_mean + 1.96*cf_se, true_ate],
})
comp["bias"] = comp["estimate"] - true_ate
print(comp.to_string(index=False, float_format=lambda v: f"{v:7.3f}"))

# Forest plot
fig, ax = plt.subplots(figsize=(9, 4.5))
y_pos = np.arange(len(comp))[::-1]
fp_colors = [COLOR_GRAY, COLOR_BLUE, COLOR_TEAL, COLOR_ORANGE]
for i, (_, row) in enumerate(comp.iterrows()):
    yp = y_pos[i]
    if row["method"] == "Truth":
        ax.scatter(row["estimate"], yp, color=fp_colors[i], s=200,
                   marker="*", zorder=3, edgecolor=COLOR_BLACK, linewidth=0.8)
    else:
        ax.errorbar(row["estimate"], yp,
                    xerr=[[row["estimate"] - row["ci_low"]],
                          [row["ci_high"] - row["estimate"]]],
                    fmt="o", color=fp_colors[i], elinewidth=2, capsize=5,
                    markersize=10, markeredgecolor=COLOR_BLACK)
ax.axvline(true_ate, color=COLOR_ORANGE, linewidth=1.0, linestyle="--", alpha=0.6)
ax.set_yticks(y_pos)
ax.set_yticklabels(comp["method"])
ax.set_xlabel("Estimated ATE")
ax.set_title("ATE estimates with 95% CIs vs the truth")
plt.tight_layout()
plt.show()


The naive interval sits entirely below the truth (visible confounding bias). DoubleML's interval straddles the truth -- the only one that does. The CausalForestDML mean-of-IATEs interval is the *tightest* but its upper bound (~5.50) doesn't cover 5.628 -- it captures sampling uncertainty in the *average of individual predictions*, not in the population ATE itself.

> **Practical rule:** use **DoubleML for ATE inference**; reserve **CausalForestDML for ranking and heterogeneity**.

## Step 7 -- A welfare-maximising assignment rule

Suppose training has a fixed cost equivalent to **4 months** of employment per jobseeker. The optimal rule (if you knew the truth) would treat person $i$ if and only if $\tau_i > c$. We don't know $\tau_i$ in practice, so we plug in $\hat{\tau}_i$ from the causal forest. We benchmark four rules: **treat none**, **treat all**, **IATE rule** (estimated effect > cost), and an **oracle** that has access to the true $\tau$.

Welfare under any rule:

$$W(\text{rule}) = E[\text{rule}(X) \cdot (\tau(X) - c)]$$


In [ ]:
COST = 4.0

assign_treat_none = np.zeros(len(df), dtype=int)
assign_treat_all  = np.ones(len(df),  dtype=int)
assign_iate_rule  = (iate_hat            > COST).astype(int)
assign_oracle     = (truth["tau"].values > COST).astype(int)

def welfare(rule, tau, cost):
    return float((rule * (tau - cost)).mean())

policy = pd.DataFrame({
    "rule": ["Treat none", "Treat all",
             "IATE rule (treat where iate_hat > cost)",
             "Oracle (treat where true tau > cost)"],
    "share_treated": [assign_treat_none.mean(), assign_treat_all.mean(),
                      assign_iate_rule.mean(),  assign_oracle.mean()],
    "avg_welfare":   [welfare(assign_treat_none, truth["tau"].values, COST),
                      welfare(assign_treat_all,  truth["tau"].values, COST),
                      welfare(assign_iate_rule,  truth["tau"].values, COST),
                      welfare(assign_oracle,     truth["tau"].values, COST)],
})
print(policy.to_string(index=False, float_format=lambda v: f"{v:7.3f}"))

# Bar chart
fig, ax = plt.subplots(figsize=(9, 5.5))
pol_colors = [COLOR_GRAY, COLOR_BLUE, COLOR_TEAL, COLOR_ORANGE]
bars = ax.bar(np.arange(len(policy)), policy["avg_welfare"],
              color=pol_colors, edgecolor=COLOR_BLACK, width=0.6)
y_max = float(policy["avg_welfare"].max()) * 1.30
ax.set_ylim(top=y_max)
for bar, val, sh in zip(bars, policy["avg_welfare"], policy["share_treated"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + y_max * 0.03,
            f"{val:.2f}\n({sh*100:.0f}% treated)",
            ha="center", fontsize=9)
ax.axhline(0, color=COLOR_BLACK, linewidth=0.8)
ax.set_xticks(np.arange(len(policy)))
ax.set_xticklabels(["Treat\nnone", "Treat\nall", "IATE\nrule", "Oracle"])
ax.set_ylabel(f"Avg net welfare per person\n(true τ − cost = {COST:.0f} months)")
ax.set_title("Welfare under alternative training-assignment rules")
plt.tight_layout()
plt.show()


The IATE rule trains **83.9%** of the cohort -- almost identical to the oracle's **83.8%** -- and lifts welfare to **1.749 months per person**. That's **99.5% of the oracle's 1.758** and **7.4% above treat-all**. A simple thresholding rule built on $\hat{\tau}$ recovers nearly all the welfare gain available from perfect knowledge.

## Takeaways

- **Naive comparisons are biased on observational data.** 5.111 vs true 5.628 — the 95% CI fails to cover the truth.
- **DoubleML closes 79% of the bias** and delivers correct CI coverage. Use it for ATE.
- **Causal forests recover individual effects with correlation 0.956 and 0.40-month MAE.** Use them for ranking and heterogeneity.
- **A simple IATE-thresholding rule recovers 99.5% of oracle welfare** -- concrete evidence that CML is not just academic.
- **DoubleML's CI for the ATE is not interchangeable with the forest's CI for the average of IATEs.** They target different quantities.

## Try it yourself

- Re-run Step 7 with `COST = 2.0` and `COST = 6.0`. How does the IATE rule's share-treated change? At what cost does the rule converge to "treat all" or "treat none"?
- Swap the nuisance learners for `LassoCV` and `LogisticRegressionCV`. Does the ATE estimate change meaningfully? Does the 95% CI still cover the truth?
- Compute the IATE at $X$ profiles that differ *only* in `migrant` (median values for the other 5 covariates). Does the forest predict a clear migrant effect?

## References

- Lechner, M. (2023). [*Causal Machine Learning and its use for public policy.*](https://doi.org/10.1186/s41937-023-00113-y) Swiss Journal of Economics and Statistics.
- Cockx, B., Lechner, M. & Bollens, J. (2023). [*Priority to unemployed immigrants? A causal machine learning evaluation of training in Belgium.*](https://doi.org/10.1016/j.labeco.2023.102306) Labour Economics.
- Chernozhukov et al. (2018). [*Double/debiased machine learning for treatment and structural parameters.*](https://doi.org/10.1111/ectj.12097) The Econometrics Journal.
- Athey, S., Tibshirani, J. & Wager, S. (2019). [*Generalized random forests.*](https://doi.org/10.1214/18-AOS1709) Annals of Statistics.
- [DoubleML](https://docs.doubleml.org/) and [EconML](https://econml.azurewebsites.net/) Python packages.
- Full blog post: [carlos-mendez.org/post/python_cml/](https://carlos-mendez.org/post/python_cml/)
